# LoRA Instruction Tuning, Measured with IFEval

Fine-tuning TinyLlama-1.1B for instruction *following* — not for knowledge — with
LoRA, then measuring it on IFEval, a benchmark of verifiable constraints ("write
it in all caps", "avoid the word X", "use exactly 3 bullet points") that can be
checked programmatically rather than judged.

**Result: a 3.5–4.3x improvement across all four IFEval metrics, from training
2.24% of the model's parameters.**

| Metric | Base | LoRA-tuned | Growth |
|---|---|---|---|
| Prompt-level strict | 3.51% | **15.16%** | 4.3x |
| Instruction-level strict | 7.55% | **27.34%** | 3.6x |
| Prompt-level loose | 4.25% | **16.45%** | 3.9x |
| Instruction-level loose | 8.15% | **28.78%** | 3.5x |

The absolute numbers stay low — 15% of prompts fully satisfied is not a usable
assistant — but that is the right expectation for a 1.1B model on IFEval. The
finding is the *ratio*: constraint-following is a learnable behaviour that a
small adapter trained on 2,000 examples can substantially install.

## 1. An instruction-following dataset

`allenai/tulu-3-sft-personas-instruction-following`, 2,000 samples. Every example
pairs a prompt carrying an explicit, checkable constraint with a response that
satisfies it — the same kind of constraint IFEval scores, which is what makes
this dataset the right training signal for this benchmark.

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset_name = "allenai/tulu-3-sft-personas-instruction-following"
sample_size = 2000

ds = load_dataset(dataset_name, split="train")

ds_subset = ds.select(range(sample_size))

print(f"Loaded {len(ds_subset)} samples.")

sample = ds_subset[0]
for key, value in sample.items():
    print(f"[{key}]:\n{value}\n")

df = pd.DataFrame(ds_subset)

## 2. The chat template

Rendering the tokenizer's chat template to see the exact string the model will be
trained on. Getting this wrong is the most common silent failure in instruction
tuning: train on `<|user|>` when the model expects a different turn marker and
the loss curve looks fine while the tuned model never recognises a turn boundary
at inference.

In [ ]:
import os
from IPython.display import display, HTML
from transformers import AutoTokenizer
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

COLORS = [
    "#D6EAF8", "#D5F5E3", "#FCF3CF", "#FADBD8", 
    "#EBDEF0", "#E8DAEF", "#D4E6F1", "#EAFAF1"
]

sample_text1 = """English and CAPITALIZATION 🎵 漢字
show_tokens False None elif == >= else:
two tabs:"	" Three tabs: "			"
3.25 * 12 = 39"""

sample_text2 = "من علیرضا شیری دانشجوی کارشناسی ارشد هوش مصنوعی هستم و در حال گذراندن درس پردازش زبان طبیعی هستم."

model_names = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "google/gemma-2-2b-it",
    "meta-llama/Llama-3.1-8B"
]

for model_id in model_names:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    token_ids1 = tokenizer.encode(sample_text1, add_special_tokens=False)
    
    html_parts = [
        '<div style="margin: 20px 0; padding: 20px; background-color: #ffffff; '
        'border: 1px solid #e0e0e0; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">'
        f'<h3 style="margin-top: 0; color: #2c3e50; font-family: sans-serif; border-bottom: 2px solid #3498db; display: inline-block; padding-bottom: 5px;">{model_id} - Sample 1</h3>'
        '<div style="line-height: 2.2; margin-top: 15px;">'
    ]

    for i, t_id in enumerate(token_ids1):
        color = COLORS[i % len(COLORS)]
        decoded_text = tokenizer.decode([t_id])
        display_token = decoded_text.replace("<", "&lt;").replace(">", "&gt;").replace("\n", "\\n")
        display_token = display_token.replace(' ', '&nbsp;')
        
        if not display_token:
            display_token = ""

        style = (
            f"background-color: {color}; color: #333; padding: 4px 8px; "
            f"margin: 0 3px; border-radius: 6px; font-family: 'Consolas', monospace; "
            f"font-size: 14px; border: 1px solid #bdc3c7; display: inline-block;"
        )
        html_parts.append(f'<span style="{style}">{display_token}</span>')

    html_parts.append('</div></div>')
    display(HTML("".join(html_parts)))


    token_ids2 = tokenizer.encode(sample_text2, add_special_tokens=False)
    
    html_parts_2 = [
        '<div style="margin: 20px 0; padding: 20px; background-color: #ffffff; '
        'border: 1px solid #e0e0e0; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">'
        f'<h3 style="margin-top: 0; color: #2c3e50; font-family: sans-serif; border-bottom: 2px solid #3498db; display: inline-block; padding-bottom: 5px;">{model_id} - Sample 2 (Persian)</h3>'
        '<div style="line-height: 2.5; margin-top: 15px; direction: rtl;">'
    ]

    for i, t_id in enumerate(token_ids2):
        color = COLORS[i % len(COLORS)]
        decoded_text = tokenizer.decode([t_id])
        display_token = decoded_text.replace("<", "&lt;").replace(">", "&gt;")
        display_token = display_token.replace(' ', '&nbsp;')

        if not display_token:
            display_token = ""

        style = (
            f"background-color: {color}; color: #333; padding: 4px 8px; "
            f"margin: 0 3px; border-radius: 6px; font-family: 'Tahoma', sans-serif; "
            f"font-size: 14px; border: 1px solid #bdc3c7; display: inline-block;"
        )
        html_parts_2.append(f'<span style="{style}">{display_token}</span>')

    html_parts_2.append('</div></div>')
    display(HTML("".join(html_parts_2)))

## 3. The base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model and tokenizer loaded successfully.")
print(f"Device used: {model.device}")

### Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "You are my assistant."},
    {"role": "user", "content": "Hello! What can you do?"}
]

formatted_chat = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("Structured Chat Prompt:\n")
print(formatted_chat)

## 4. Formatting the training set

Mapping each example through the chat template into a single `text` field.

In [ ]:
MAX_SEQ_LENGTH = 2048

train_dataset = ds_subset.map(
    lambda example: {
        "text": tokenizer.apply_chat_template(
            example["messages"], 
            tokenize=False, 
            add_generation_prompt=False
        )
    }
)

print("Sample processed text:")
print(train_dataset[0]["text"])

## 5. LoRA configuration and training

Rank 32, alpha 64, dropout 0.05, applied to all seven projection matrices
(`q/k/v/o` in attention and `gate/up/down` in the MLP). Five epochs at 2e-4 with
gradient accumulation to an effective batch size of 16.

**25,231,360 trainable parameters out of 1,125,279,744 — 2.24%.**

Targeting the MLP projections as well as attention is a deliberate choice. Common
LoRA practice touches only `q_proj` and `v_proj`; including `gate/up/down` roughly
triples the adapter size, and for *format* compliance — closer to a stylistic
behaviour than a factual one — the MLP is where it pays off.

Weights are merged back into the base model afterwards so the evaluation harness
can load it as an ordinary checkpoint.

In [ ]:
import torch
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType


peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model.enable_input_require_grads()
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


def tokenize_function(examples):
    tokenized_output = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=1024  
    )
    tokenized_output["labels"] = tokenized_output["input_ids"].copy()
    return tokenized_output

print("Tokenizing dataset")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
columns_to_remove = [col for col in tokenized_train_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
tokenized_train_dataset = tokenized_train_dataset.remove_columns(columns_to_remove)


training_args = TrainingArguments(
    output_dir="./tinyllama_improved_results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_torch",
    report_to="none"
)


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
)

print("Starting training.")
trainer.train()

print("Saving final model")
adapter_path = "./tinyllama_improved_adapter"
trainer.save_model(adapter_path)

print("Merging weights")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./tinyllama_improved_merged")
tokenizer.save_pretrained("./tinyllama_improved_merged")

print(" Model saved to './tinyllama_improved_merged'")

### Saving the adapter

In [ ]:
import os

adapter_path = "./tinyllama_final_adapter"
trainer.save_model(adapter_path)

merged_model = model.merge_and_unload()

merged_path = "./tinyllama_final_merged"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size

adapter_size = get_folder_size(adapter_path) / (1024 * 1024)
merged_size = get_folder_size(merged_path) / (1024 * 1024)

print(f"Adapter Size: {adapter_size:.2f} MB")
print(f"Merged Model Size: {merged_size:.2f} MB")

## 6. Base vs. tuned, side by side

The clearest evidence in the notebook, and it needs no metric at all. Two prompts,
identical except for one added constraint:

**Prompt 1** — *"Create a slogan for a new vintage blues record collection app."*
Both models answer well. No constraint, no difference.

**Prompt 2** — the same prompt plus *"The slogan should written in all capital
letters."*

- **Base:** `"Discover the Sound of Blues: A Journey Through Time and Tradition."`
  — a good slogan that ignores the instruction entirely.
- **Tuned:** `UNCOVER THE ROOTS, RECORD THE SOUL!` — in caps.

The base model is not worse at slogans; it is worse at *noticing that a constraint
was imposed*. That gap is exactly what IFEval quantifies below, and it is why the
base model scores 3.51% at prompt level despite producing fluent text throughout.

The tuned model is also visibly more verbose, continuing past the slogan into
unrequested explanation — a known side effect of SFT on long-form responses, and a
cost worth naming.

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

prompts = [
    "Create a slogan for a new vintage blues record collection app.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters and end with the word soul.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters, end with the word soul, and contain exactly seven words.",
    "Create a slogan for a new vintage blues record collection app. The slogan should written in all capital letters, end with the word soul, and contain exactly seven words. Wrap the entire answer in JSON format."
]

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
merged_model_path = "./tinyllama_final_merged"

def generate_clean_response(model, tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, 
        return_tensors="pt", 
        add_generation_prompt=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids, 
            max_new_tokens=100, 
            do_sample=True, 
            temperature=0.6,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )
    
   
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
   
    if "<|assistant|>" in tokenizer.decode(outputs[0]): 
        
        pass 

    response = full_text.split("assistant\n")[-1] if "assistant\n" in full_text else full_text
    return response.strip()


print("LOADING BASE MODEL")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto", torch_dtype=torch.float16)
base_results = [generate_clean_response(model, tokenizer, p) for p in prompts]

del model
torch.cuda.empty_cache()
gc.collect()


print("LOADING MERGED MODEL.")
tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
model = AutoModelForCausalLM.from_pretrained(merged_model_path, device_map="auto", torch_dtype=torch.float16)
merged_results = [generate_clean_response(model, tokenizer, p) for p in prompts]

del model
torch.cuda.empty_cache()
gc.collect()


print("\n" + "."*100)
for i, (p, base, merged) in enumerate(zip(prompts, base_results, merged_results)):
    print(f"PROMPT {i+1}: {p}\n")
    print(f" BASE:\n{base}\n")
    print(f" MERGED:\n{merged}\n")
    print("."*100)

## 7. IFEval on the base model

`lm_eval` with the `ifeval` task. Four metrics: *prompt-level* requires every
constraint in a prompt to be satisfied, *instruction-level* scores each constraint
separately, and the *loose* variants allow minor formatting slack (e.g. surrounding
markdown) that *strict* rejects.

**Base: 3.51% / 7.55% / 4.25% / 8.15%.**

In [ ]:
import lm_eval
import torch
import gc
import json

def run_evaluation(model_path):
    gc.collect()
    torch.cuda.empty_cache()
    
    try:
        results = lm_eval.simple_evaluate(
            model="hf",
            model_args={
                "pretrained": model_path,
                "dtype": "float16",
                "trust_remote_code": True
            },
            tasks=["ifeval"],
            batch_size="auto",
            device="cuda"
        )
        if 'results' in results and 'ifeval' in results['results']:
            return results['results']['ifeval']
    except Exception as e:
        print(f"Error: {e}")
    return {}

model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Evaluating: {model_path}")
metrics = run_evaluation(model_path)

print("-" * 30)
print(f"Base Model Results:")
for k, v in metrics.items():
    if k in ["prompt_level_strict_acc", "inst_level_strict_acc"]:
        print(f"{k}: {v:.4f}")

with open('base_metrics.json', 'w') as f:
    json.dump(metrics, f)

In [ ]:
import json


print(f"Status of metrics: {type(metrics)}")
if hasattr(metrics, 'keys'):
    print("All available keys:", list(metrics.keys()))
    
    print("\n" + "-"*30)
    print("FULL RESULTS DUMP:")

    for k, v in metrics.items():
        print(f"{k}: {v}")
else:
    print("Metrics variable is empty or not a dictionary!", metrics)

with open('base_metrics_debug.json', 'w') as f:

    safe_dump = {str(k): float(v) if isinstance(v, (int, float)) else str(v) for k, v in metrics.items()}
    json.dump(safe_dump, f)

print("\n" + "="*30)
print(" Backup saved to 'base_metrics_debug.json'")

## 8. IFEval on the tuned model

**Tuned: 15.16% / 27.34% / 16.45% / 28.78%.**

In [ ]:
import lm_eval
import torch
import gc
import json

def run_evaluation(model_path):
    gc.collect()
    torch.cuda.empty_cache()
    
    try:
        results = lm_eval.simple_evaluate(
            model="hf",
            model_args={
                "pretrained": model_path,
                "dtype": "float16",
                "trust_remote_code": True
                
            },
            tasks=["ifeval"],
            batch_size="8",
            device="cuda"
        )
        if 'results' in results and 'ifeval' in results['results']:
            return results['results']['ifeval']
    except Exception as e:
        print(f"Error: {e}")
    return {}

model_path = "./TinyLlama_Merged_Full"
print(f"Evaluating: {model_path}")
metrics = run_evaluation(model_path)

print("-" * 30)
print(f"Merged Model Results:")
for k, v in metrics.items():
    if k in ["prompt_level_strict_acc", "inst_level_strict_acc"]:
        print(f"{k}: {v:.4f}")

with open('merged_metrics.json', 'w') as f:
    json.dump(metrics, f)

In [ ]:
import json


print(f"Status of metrics: {type(metrics)}")
if hasattr(metrics, 'keys'):
    print("All available keys:", list(metrics.keys()))
    
    print("\n" + "-"*30)
    print("FULL RESULTS DUMP:")

    for k, v in metrics.items():
        print(f"{k}: {v}")
else:
    print("Metrics variable is empty or not a dictionary!", metrics)

## 9. The comparison

| Metric | Base Model | Fine-Tuned | Improvement | Growth |
|---|---|---|---|---|
| Prompt Level Strict Acc | 3.51% | 15.16% | +11.65 pp | 4.3x |
| Inst Level Strict Acc | 7.55% | 27.34% | +19.78 pp | 3.6x |
| Prompt Level Loose Acc | 4.25% | 16.45% | +12.20 pp | 3.9x |
| Inst Level Loose Acc | 8.15% | 28.78% | +20.62 pp | 3.5x |

Two things worth reading out of this table beyond the headline ratio:

- **Instruction-level is consistently ~1.8x prompt-level** for both models. The
  tuned model satisfies 27% of individual constraints but only 15% of prompts
  completely, because prompt-level is conjunctive — a prompt with three
  constraints needs all three. Multi-constraint prompts are where it still fails.
- **The strict/loose gap is small** (~1.3 pp) and does not widen after tuning. The
  failures are substantive, not formatting noise: the tuned model is not losing
  points to stray markdown, it is missing constraints outright.

In [ ]:
import pandas as pd


data = {
    "Metric": [
        "Prompt Level Strict Acc",
        "Inst Level Strict Acc", 
        "Prompt Level Loose Acc",
        "Inst Level Loose Acc"   
    ],
    "Base Model": [0.035120, 0.075540, 0.042514, 0.081535],
    "Fine-Tuned Model": [0.151571, 0.273381, 0.164510, 0.287770]
}


df = pd.DataFrame(data)


df["Improvement"] = df["Fine-Tuned Model"] - df["Base Model"]


df["Growth (x)"] = df["Fine-Tuned Model"] / df["Base Model"]


formatted_df = df.copy()
cols_to_percent = ["Base Model", "Fine-Tuned Model", "Improvement"]
for col in cols_to_percent:
    formatted_df[col] = formatted_df[col].apply(lambda x: f"{x:.2%}")

formatted_df["Growth (x)"] = formatted_df["Growth (x)"].apply(lambda x: f"{x:.1f}x")


print(formatted_df.to_markdown(index=False))